# Classical Object Detectors vs. MLLMs for Child Detection in Images: A Preliminary Study

Run experiments with 3 approches for child detection:

1. **Classical person detector + age estimation** (YOLO26x + MiVOLO)
2. **MLLM-based child localization** — obtain detections via local vision LLM
3. **MLLM-based age estimation** — classical person detector with age estimation obtained via local LLM

Before running the notebook, set environmental variables `IMAGESDIR`, `OUTPUTDIR`, and `MODELSDIR`. For experiments 2–3, start `llama-server` first (see `llama_helper.md`).

In [1]:
import os
from prettytable import PrettyTable
from pycocotools.coco import COCO

# --- configure paths (override with env if already set) ---
IMAGESDIR = os.environ.get("IMAGESDIR")
OUTPUTDIR = os.environ.get("OUTPUTDIR")
MODELSDIR = os.environ.get("MODELSDIR")
GT_PATH = "../../data/parenting_slop_dataset/annotations/parenting_slop_dataset.json"
LLM_PORT = "8010"

os.makedirs(OUTPUTDIR, exist_ok=True)

print("Directory with frames: "+IMAGESDIR)

detection_engine_name = "detector_yolo26x"
llm_engine_name = "llm_Qwen3.6-27B"

EXP0 = {"engine": detection_engine_name, "ae": ""}
EXP1 = {"engine": detection_engine_name, "ae": "mivolo"}
EXP2 = {"engine": llm_engine_name, "ae": ""}
EXP3 = {"engine": detection_engine_name, "ae": llm_engine_name}



Directory with frames: /home/weronika/Dane/sandbox


## Shared step for experiment 1 and 3: person detection with YOLO26x

Used by experiments 1 and 3. Detects all the people in images from IMAGESDIR with YOLO26x and write the results in `*_all_detections.json` under `OUTPUTDIR/detector_yolo26x/`.

In [3]:

from run_detection_with_yolo import run_detection as run_detection_with_yolo
detection_engine_weights = os.path.join(MODELSDIR,"yolo26x.pt")

run_detection_with_yolo(IMAGESDIR, OUTPUTDIR, detection_engine_weights)


Processing file: /home/weronika/Dane/sandbox/0001/frame000012.jpg
Detected 1 people
Processing file: /home/weronika/Dane/sandbox/0001/frame000000.jpg
Detected 1 people
Processing file: /home/weronika/Dane/sandbox/0001/frame000006.jpg
Detected 1 people
Processing file: /home/weronika/Dane/sandbox/0000/frame000012.jpg
Detected 1 people
Processing file: /home/weronika/Dane/sandbox/0000/frame000000.jpg
Detected 2 people
Processing file: /home/weronika/Dane/sandbox/0000/frame000006.jpg
Detected 1 people
Correct JSONS in 6/6 results. Avg processing time: 0.1586s over 6 files.


## Experiment 1: Classical person detector + age estimation

Reads detections and estimate age for each bounding box with person using MiVOLO library. Results are saved as `*_with_age_mivolo.json`. It requires installing MiVOLO.

In [2]:
from age_estimation.run_age_estimation_with_mivolo import run_age_estimation as run_age_estimation_with_mivolo
run_age_estimation_with_mivolo(IMAGESDIR, OUTPUTDIR, detection_engine_name)

print("Experiment 1 done:", EXP1)


/home/weronika/Dev/Python-envs/MinorsReportSandbox2/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Processing /home/weronika/Output/MINORS_REPORT/sandbox/detector_yolo26x/0001/frame000000_all_detections.json
Processing /home/weronika/Output/MINORS_REPORT/sandbox/detector_yolo26x/0001/frame000006_all_detections.json
Processing /home/weronika/Output/MINORS_REPORT/sandbox/detector_yolo26x/0001/frame000012_all_detections.json
Processing /home/weronika/Output/MINORS_REPORT/sandbox/detector_yolo26x/0000/frame000000_all_detections.json
Processing /home/weronika/Output/MINORS_REPORT/sandbox/detector_yolo26x/0000/frame000006_all_detections.json
Processing /home/weronika/Output/MINORS_REPORT/sandbox/detector_yolo26x/0000/frame000012_all_detections.json
Correct entries JSONS in 6 results. For 7/7 bboxes age is estimated. 4 are adults, 3 are minors. 0 are uknown. Avg processing time: 0.0808s over 6 files.
Experiment 1 done: {'engine': 'detector_yolo26x', 'ae': 'mivolo'}


## Experiment 2: MLLM-based child localization

Detects children in images using LLM.  It requires `llama-server` running Qwen3.6-27B model on port 8010. Detection results are saved as `*_raw.txt`, then they are parsed and saved as `*_fixed.json`.

In [2]:

from run_detection_with_llm import run_detection as run_detection_with_llm
run_detection_with_llm(IMAGESDIR, OUTPUTDIR, LLM_PORT)
from llm_result_to_jsons import run_postprocessing
run_postprocessing(IMAGESDIR, OUTPUTDIR, llm_engine_name)

print("Experiment 2 done:", EXP2)


Correct entries JSONS in 0 results. 6 are fixable
Experiment 2 done: {'engine': 'llm_Qwen3.6-27B', 'ae': ''}


## Experiment 3: MLLM-based age estimation

 It estimates age of people with LLM in bounding boxes calculated in the first step. It requires `llama-server` running Qwen3.6-27B model on port 8010. Results are written as `*_with_age_llm_<model>.json`.

In [7]:

from age_estimation.run_age_estimation_with_llm import run_age_estimation as run_age_estimation_with_llm
run_age_estimation_with_llm(IMAGESDIR, OUTPUTDIR, detection_engine_name, LLM_PORT)

print("Experiment 3 done:", EXP3)


Correct entries JSONS in 6 results. For 7/7 bboxes age is estimated. 1 are adults, 6 are minors. 0 are uknown. Avg processing time: 0.0000s over 0 files.
Experiment 3 done: {'engine': 'detector_yolo26x', 'ae': 'llm_Qwen3.6-27B'}


## Evaluation

Uses the same COCO evaluation helpers and summary table as `evaluate_coco_detection.py`.

In [2]:

from evaluate_coco_detection import eval
CONF_TH = 0

experiments = [EXP0, EXP1, EXP2, EXP3]
coco_gt = COCO(str(GT_PATH))

summary_table = PrettyTable()
summary_table.field_names = [
    "engine",
    "age-estimation",
    "AP",
    "AP50",
    "AP75",
    "AR100",
    "global-P",
    "global-R",
    "global-F1",
]

for exp in experiments:
    engine = exp["engine"]
    ae = exp["ae"]
    suffix  = "_with_age"+(("_"+ae) if ae != "" else "")+ ".json" if ("detector" in engine  and ae!="") else "_fixed.json"

    row = eval(OUTPUTDIR,engine,ae,coco_gt,suffix)
    summary_table.add_row(row)

print("\nSummary:")
print(summary_table)


loading annotations into memory...
Done (t=0.00s)
creating index...
index created!
GT images: 1000
GT annotations: 1975
Predictions: 7 boxes from 6/6 files (0 without GT match, 0 bad JSON)
Wrote predictions to /home/weronika/Output/MINORS_REPORT/sandbox/detector_yolo26x/detresults.json

COCO bbox metrics for detector_yolo26x:
Loading and preparing results...
DONE (t=0.00s)
creating index...
index created!
Running per image evaluation...
Evaluate annotation type *bbox*
DONE (t=0.14s).
Accumulating evaluation results...
DONE (t=0.02s).
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=   all | maxDets=100 ] = 0.010
 Average Precision  (AP) @[ IoU=0.50      | area=   all | maxDets=100 ] = 0.010
 Average Precision  (AP) @[ IoU=0.75      | area=   all | maxDets=100 ] = 0.010
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= small | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area=medium | maxDets=100 ] = 0.000
 Average Precision  (AP) @[ IoU=0.50:0.95 | area= large | ma